In [48]:
import math
import torch
import torch.nn.functional as F

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # d_k: 각 토근 벡터의 차원 (Q의 마지막 차원 크기)
    # scaling에 사용된다. 값이 너무 커지는 것을 방지
    d_k = Q.size(-1)
    
    # K를 transpose: (batch, seq_len, d_k) -> (batch, d_k, seq_len)
    # K를 전치하여 Q와 행렬곱이 가능하도록 변환
    K_t = K.transpose(1, 2)

    # Qeury와 Key의 내적을 통해 토큰 간 유사도(score) 계산
    # 결과 크기: (batch, seq_len, seq_len)
    scores = torch.matmul(Q, K_t)

    # score가 너무 커져 Softmax가 포화되는 걸 방지하기 위해
    # d_k로 나누어 Scaling
    scores = scores / math.sqrt(d_k)
    
    # Decoder의 Masked Attention에서 사용
    # 미래 토큰이나 Padding 토큰을 보지 못하도록
    # mask가 0인 위치를 -inf로 만들어 Softmax결과가 0이 되게 함
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # 유사도(score)를 확률(Attention Weight)로 변환
    # 각 Query가 어떤 Key를 얼마나 참고할지 결정
    attn_weights = F.softmax(scores, dim=-1)

    # Attention Weight를 Value에 곱하여
    # 중요한 정보는 많이 덜 중요한 정보는 적게 반영
    output = torch.matmul(attn_weights, V)

    # score       : Softmax 적용 전 유사도 점수
    # attn_weights: Softmax 적용 후 Attention 가중치
    # output      : 최종 Attention 결과
    return scores, attn_weights, output


In [50]:
# 2: batch_size
# 4: seq_len 
# 8: d_k
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

mask = torch.tril(torch.ones(4, 4))

scores, attn_weights, output = scaled_dot_product_attention(Q, K, V, mask=mask)

print("scores:", scores.shape)
print("attn_weights:", attn_weights.shape)
print("output:", output.shape)
print(attn_weights[0])

scores: torch.Size([2, 4, 4])
attn_weights: torch.Size([2, 4, 4])
output: torch.Size([2, 4, 8])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.9675, 0.0325, 0.0000, 0.0000],
        [0.0962, 0.6959, 0.2079, 0.0000],
        [0.0833, 0.0175, 0.0495, 0.8497]])
